In [1]:
import torch
import gc
from sentence_transformers import SentenceTransformer
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

/home/damian/New Folder/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
wiki_plots = load_dataset("vishnupriyavr/wiki-movie-plots-with-summaries", split='train')
wiki_plots

Dataset({
    features: ['Release Year', 'Title', 'Origin/Ethnicity', 'Director', 'Cast', 'Genre', 'Wiki Page', 'Plot', 'PlotSummary'],
    num_rows: 34886
})

In [13]:
print(wiki_plots['Title'][0], ": :", wiki_plots['PlotSummary'][0])

Kansas Saloon Smashers : : Carrie Nation and her followers burst into a saloon and attack a bartender. The group then begin wrecking the bar, smashing the fixtures, mirrors, and breaking the cash register. The bartender sprays seltzer water in Nation's face before a group of policemen appear and order everybody to leave.


In [10]:
plot_summaries = [wiki_plot for wiki_plot in wiki_plots['PlotSummary']]
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
plot_encodings = sbert_model.encode(plot_summaries, convert_to_tensor=True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13105.61it/s]


In [11]:
def find_similarities(description, top_k=3):
    description_encoding = sbert_model.encode([description], convert_to_tensor=True)
    similarities = sbert_model.similarity(description_encoding, plot_encodings)
    sim_indices = similarities.topk(top_k).indices
    return [wiki_plots['Title'][i] for i in sim_indices]

In [44]:
find_similarities('ancient roman empire')

[['Quo Vadis', 'Sign of the Pagan', 'The Fall of the Roman Empire']]

# #4

In [5]:
qwen_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-7B-Instruct')
qwen_model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-7B-Instruct', device_map='auto', dtype='auto')

Loading weights: 100%|██████████| 339/339 [00:02<00:00, 129.12it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


In [12]:
def generate(model, tokenizer, prompt, max_new_tokens=50, **generate_kwargs):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id, **generate_kwargs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

class MovieExpert:
    def __init__(self, model, tokenizer, max_answer_length=10000):
        self.context = """You are Assistant. Assistant is a movie expert.

You have access to a movie database through RAG.

Whenever the user asks for a movie recommendation, you MUST request
information from the database using exactly this format:

<RAG_START>concise search query<RAG_END>

Example 1: \nUser: Hi, how are you? could you recommend me some scary movies about zombies?\nAssistant: Hi, I am great, thanks! what about you? Sure, there some movies I'd like to recommend you <RAG_START>scary movie about zombies<RAG_END>
Example 2: \nUser: recommend me a film about world war 2\nAssistant: Sure! Here are some films about world war 2: <RAG_START> movie world war 2 <RAG_END>
Example 3: \nUser: Hello! can you advice me something to watch about cars and races?\nAssistant: Hello!, Of course I can!\nHere are my recommendations: <RAG_START> cars and races <RAG_END>

Do not answer the recommendation until the database information has
been provided.

For all other questions, answer normally."""

        self.model = model
        self.tokenizer = tokenizer
        self.max_answer_length = max_answer_length
        self.rag_flag = 0
        
    def chat(self, prompt):
        self.context += "\nUser: " + prompt + "\nAssistant:"
        context = self.context
        start_index = len(context)
        while True:
            extended = generate(self.model, self.tokenizer, context, max_new_tokens=50)
            answer = extended[start_index:]
            
            if "<RAG_START>" in answer and "<RAG_END>" in answer:
                request = answer.split('<RAG_START>')[1].split('<RAG_END>')
                if type(request) is list:
                    request = request[0]
                #print(request)
                movies_lst = find_similarities(request)
                #print(movies_lst[0])
                movies = "".join(i + ", " for i in movies_lst[0])
                answer = answer.replace('<RAG_START>' + request + '<RAG_END>', movies)
                self.rag_flag += 1
            
            if ('User:' in answer or extended==context or len(answer)>self.max_answer_length): break
            context = extended
        answer = answer.split('User:')[0]
        self.context += answer
        return answer.strip()

In [2]:
gc.collect()
torch.cuda.empty_cache()

In [3]:
qwen_tokenizer_2 = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct')
qwen_model_2 = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct', device_map='cuda', dtype='auto')

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 953.99it/s]


In [4]:
few_shot_learning = ["\nUser: Hi, how are you? could you recommend me some scary movies about zombies?\nAssistant: Hi, I am great, thanks! what about you? Sure, there some movies I'd like to recommend you <RAG_START>scary movie about zombies<RAG_END>",
                     "\nUser: recommend me a film about world war 2\nAssistant: Sure! Here are some films about world war 2: <RAG_START> movie world war 2 <RAG_END>",
                     "\nUser: Hello! can you advice me something to watch about cars and races?\nAssistant: Hello!, Of course I can!\nHere are my recommendations: <RAG_START> cars and races <RAG_END>"]
few_shot_learning = Dataset.from_dict({'text' : few_shot_learning})

In [21]:
sft_output_dir = './sft_ex3'
sft_config = SFTConfig(sft_output_dir, per_device_train_batch_size=4, num_train_epochs=300)
peft_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'], task_type='CAUSAL_LM')
sft_train = SFTTrainer(qwen_model_2, sft_config, processing_class=qwen_tokenizer_2, train_dataset=few_shot_learning, peft_config=peft_config)

/home/damian/New Folder/.venv/lib/python3.14/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/damian/New Folder/.venv/lib/python3.14/site-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Truncating train dataset: 100%|██████████| 3/3 [00:00<00:00, 915.19 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 3/3 [00:00<00:00, 1459.73 examples/s]


In [22]:
train_output = sft_train.train()
sft_train.model.save_pretrained(sft_output_dir)

Step,Training Loss
10,3.980431
20,3.836953
30,3.650161
40,3.450914
50,3.254551
60,3.037860
70,2.817383
80,2.611090
90,2.408665
100,2.207106


In [23]:
qwen_model = AutoModelForCausalLM.from_pretrained(sft_output_dir)

Loading weights: 100%|██████████| 224/224 [00:00<00:00, 9694.41it/s]


In [24]:
test_movie_expert = MovieExpert(qwen_model, qwen_tokenizer_2)

In [26]:
print(test_movie_expert.chat('tell me which is the very good movie about superheros who save the world?'))

Certainly! Here are some highly recommended movies about superheroes saving the world: The Return of Captain Invincible, The Crimson Charm, Return of Mr. Superman,


In [28]:
print(test_movie_expert.chat("what is the best movie about crime, bands, drugs and other action"))

To recommend you the best movie about crime, gangs, drugs and other action, I would need to perform a more specific search. Could you please provide me with a concise search query that focuses on these elements? For example:
<search query here>  
RAG_START
best movie about crime,RAG_END
RAG_START
gangs,
drugs,RAG_END
RAG_START
action,RAG_END
RAG_START
crime,RAG_END
RAG_END
Please let me know so I can proceed with your request. Assistant

Note: Remember to always provide a precise search query when asking for recommendations in this manner. The assistant will then use it to fetch relevant information from the database.


In [30]:
print(test_movie_expert.chat("ok, and the last one advice me anything related to the ancient Roman Empire"))

Sure! Here are some movies related to the Ancient Roman Empire: Quo Vadis, Sign of the Pagan, The Fall of the Roman Empire,  Assistant


In [31]:
test_movie_expert.rag_flag

4